In [ ]:
# align with miniprot
from Bio import SeqIO
from Bio.Seq import Seq
from pathlib import Path
import re
import os
import sys
def get_proportion(paf_file):
    """
    Get proportion of the sequence that is mapped by the aa probe with miniprot
    """
    with open(paf_file, 'r') as f:
        for line in f:
            if not line.startswith('##ATN'):
                continue
            sequence = line.replace('##ATN\t','').strip().replace('-','')
            return (sum(1 for c in sequence if c.isupper())/ len(sequence))

# parse the mega probes (proteins) into a dictionary
probes = SeqIO.to_dict(SeqIO.parse('/home/yjkbertrand/Documents/projects/fumaria_sim/azurella_orfipy/out_test/probes_mega_azorella_aa.fasta', 'fasta'))

#where to find the tables and fastas
%cd /home/yjkbertrand/Documents/projects/nextpyper_simulation/tribbles
#parse tables into a dictionary
table_dict = {}
for table in Path.cwd().glob('*.tsv'):
    with open(table) as f:
        next(f)
        for line in f:
            splt = line.split('\t')
            table_dict[splt[0]] = splt[1]
proportions = []

# loop of the fasta files
for fasta in sorted(Path.cwd().glob('probes_5335_p*_unaln.fasta')):
    paralogs = SeqIO.parse(fasta, 'fasta')
    fasta_name = fasta.stem
    for idx, paralog in enumerate(list(paralogs)):
        new_record = []
        paralog.seq = Seq(str(paralog.seq).replace('-',''))
        new_record.append(paralog)
        new_paralog_id = re.sub(r"-\d+\_EDGE", "-EDGE", paralog.id)
        probe_id = table_dict[new_paralog_id]
        if 'grafted' in probe_id:
            new_probe_id = probe_id.replace('_grafted_with_', '_grafted_with-')
        else:
            new_probe_id = probe_id.replace('_','-')
        probe = [probes[new_probe_id]]
        SeqIO.write(probe, "probe_tmp.fasta", "fasta")
        SeqIO.write([paralog], f"paralog_tmp.fasta", "fasta")
        !miniprot -t 8 --gff --outn 1 -J 50 -E 3 --aln  paralog_tmp.fasta probe_tmp.fasta > miniprot_tmp.paf

        proportion_matches = get_proportion('miniprot_tmp.paf')
        for p in Path.cwd().glob('*tmp*'):
            p.unlink()
        proportions.append(f"{paralog.id},{new_probe_id},{len(paralog.seq)},{len(probe[0].seq)*3},{proportion_matches}\n")

with open("proportion_table_all.csv", "w") as f:
    f.write("scaffold,probe,scaffold_length,probe_length,proportion\n")
    for item in proportions:
        f.write(item)
